In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split

# 1. GENERATE MOCK COMMERCIAL PROPERTIES (JAKARTA AREA)
np.random.seed(42)
n_properties = 250

lons = np.random.uniform(106.78, 106.90, n_properties)
lats = np.random.uniform(-6.25, -6.12, n_properties)

properties_df = pd.DataFrame({
    'Property_ID': [f"PROP_{i:03d}" for i in range(n_properties)],
    'Building_Area_SqM': np.random.randint(1500, 12000, n_properties),
    'Age_Years': np.random.randint(1, 35, n_properties),
    'longitude': lons,
    'latitude': lats
})

# THE FIX: Create a logical pricing baseline (Area * Price) - (Age * Depreciation) + (Market Noise)
properties_df['Baseline_Value_USD'] = (
    (properties_df['Building_Area_SqM'] * 350)
    - (properties_df['Age_Years'] * 5000)
    + np.random.randint(-50000, 50000, n_properties)
).clip(lower=100000) # Ensure no negative property values

properties_gdf = gpd.GeoDataFrame(
    properties_df,
    geometry=gpd.points_from_xy(properties_df.longitude, properties_df.latitude),
    crs="EPSG:4326"
)

# 2. SIMULATE AN ENVIRONMENTAL HAZARD ZONE (North Jakarta Flood Plain)
flood_zone_poly = Point(106.82, -6.13).buffer(0.04)
flood_zone_gdf = gpd.GeoDataFrame(geometry=[flood_zone_poly], crs="EPSG:4326")

# 3. SPATIAL FEATURE ENGINEERING
properties_gdf['Inside_Hazard_Zone'] = properties_gdf.geometry.within(flood_zone_poly).astype(int)

mrt_hub = Point(106.824, -6.182)
properties_metric = properties_gdf.to_crs(epsg=32748)
mrt_hub_metric = gpd.GeoSeries([mrt_hub], crs="EPSG:4326").to_crs(epsg=32748).iloc[0]

properties_gdf['Distance_to_Transit_Km'] = properties_metric.geometry.distance(mrt_hub_metric) / 1000

# 4. ALGORITHMICALLY ADJUST VALUE BASED ON SPATIAL LIABILITIES
properties_gdf['True_Market_Value_USD'] = (
    properties_gdf['Baseline_Value_USD']
    + (1 / (properties_gdf['Distance_to_Transit_Km'] + 0.1) * 50000)
    - (properties_gdf['Inside_Hazard_Zone'] * (properties_gdf['Baseline_Value_USD'] * 0.20))
).astype(int)

# 5. TRAIN THE PREDICTIVE UNDERWRITING MODEL
X = properties_gdf[['Building_Area_SqM', 'Age_Years', 'Inside_Hazard_Zone', 'Distance_to_Transit_Km']]
y = properties_gdf['True_Market_Value_USD']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(f"✅ Spatial Underwriting Model Trained. Accuracy (R² Score): {model.score(X_test, y_test):.2f}")

In [ ]:
import pandas as pd
import geopandas as gpd
import numpy as np
from shapely.geometry import Point
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from google.colab import files

# 1. GENERATE COMMERCIAL PROPERTIES
np.random.seed(42)
n_properties = 250

lons = np.random.uniform(106.78, 106.90, n_properties)
lats = np.random.uniform(-6.25, -6.12, n_properties)

properties_df = pd.DataFrame({
    'Property_ID': [f"PROP_{i:03d}" for i in range(n_properties)],
    'Building_Area_SqM': np.random.randint(1500, 12000, n_properties),
    'Age_Years': np.random.randint(1, 35, n_properties),
    'longitude': lons,
    'latitude': lats
})

# Logical baseline pricing
properties_df['Baseline_Value_USD'] = (
    (properties_df['Building_Area_SqM'] * 350)
    - (properties_df['Age_Years'] * 5000)
    + np.random.randint(-50000, 50000, n_properties)
).clip(lower=100000)

properties_gdf = gpd.GeoDataFrame(
    properties_df,
    geometry=gpd.points_from_xy(properties_df.longitude, properties_df.latitude), crs="EPSG:4326"
)

# 2. SIMULATE HAZARD ZONE & SPATIAL FEATURES
flood_zone_poly = Point(106.82, -6.13).buffer(0.04)
properties_gdf['Inside_Hazard_Zone'] = properties_gdf.geometry.within(flood_zone_poly).astype(int)

mrt_hub = Point(106.824, -6.182)
properties_metric = properties_gdf.to_crs(epsg=32748)
mrt_hub_metric = gpd.GeoSeries([mrt_hub], crs="EPSG:4326").to_crs(epsg=32748).iloc[0]
properties_gdf['Distance_to_Transit_Km'] = properties_metric.geometry.distance(mrt_hub_metric) / 1000

# 3. EXPLICITLY CALCULATE ESTIMATED LOSS (The new FinTech flex)
# Assuming a 20% total value loss due to severe hazard exposure (insurance premium bump or structural damage)
properties_gdf['Estimated_Hazard_Loss_USD'] = (properties_gdf['Inside_Hazard_Zone'] * properties_gdf['Baseline_Value_USD'] * 0.20).astype(int)

# 4. FINAL VALUE CALCULATION (Baseline + Transit Premium - Hazard Loss + Market Noise)
properties_gdf['True_Market_Value_USD'] = (
    properties_gdf['Baseline_Value_USD']
    + (1 / (properties_gdf['Distance_to_Transit_Km'] + 0.1) * 50000)
    - properties_gdf['Estimated_Hazard_Loss_USD']
    + np.random.randint(-20000, 20000, n_properties) # Added market noise for realistic ML scoring
).astype(int)

# 5. ML TRAINING
X = properties_gdf[['Building_Area_SqM', 'Age_Years', 'Inside_Hazard_Zone', 'Distance_to_Transit_Km']]
y = properties_gdf['True_Market_Value_USD']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

print(f"✅ Model Trained. Realistic Accuracy (R² Score): {model.score(X_test, y_test):.2f}")
print(f"🚨 Total Portfolio Capital at Risk: ${properties_gdf['Estimated_Hazard_Loss_USD'].sum():,.2f}")

# 6. EXPORT FOR DASHBOARD
dashboard_df = properties_gdf[[
    'Property_ID', 'Building_Area_SqM', 'Age_Years', 'Distance_to_Transit_Km',
    'Baseline_Value_USD', 'Estimated_Hazard_Loss_USD', 'True_Market_Value_USD',
    'latitude', 'longitude', 'Inside_Hazard_Zone'
]].copy()

dashboard_df['Risk_Status'] = np.where(dashboard_df['Inside_Hazard_Zone'] == 1, 'High Risk (Flood/Subsidence)', 'Secure')
dashboard_df['Distance_to_Transit_Km'] = dashboard_df['Distance_to_Transit_Km'].round(2)
dashboard_df.drop(columns=['Inside_Hazard_Zone'], inplace=True)

csv_filename = "Jakarta_Commercial_Risk_Underwriting_with_Loss.csv"
dashboard_df.to_csv(csv_filename, index=False)
files.download(csv_filename)